# W06A. 집단 간 평균이 '진짜' 다른가? — t-검정과 ANOVA

> 두 집단(또는 세 집단 이상)의 평균이 '진짜' 다른지 어떻게 판단할까?

**학습 목표**
1. **독립표본 t-검정**의 논리("평균 차이 ÷ 표준오차")를 설명하고, **등분산 가정**을 확인할 수 있다.
2. **ANOVA**의 논리(집단 간 변동 vs 집단 내 변동)를 이해하고, 사후검정이 왜 필요한지 설명할 수 있다.
3. 검정 결과를 **효과 크기**(Cohen's d, η²)와 함께 해석하되, "차이가 유의하다 ≠ 차이가 크다"를 구별하는 해석문을 작성할 수 있다.

In [15]:
# 크로스플랫폼 한글 폰트 설정 (macOS / Windows)
import platform
import matplotlib.pyplot as plt

if platform.system() == "Darwin":  # macOS
    plt.rcParams["font.family"] = "AppleGothic"
elif platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

## 핵심 개념 학습

### 왜 필요한가

- **4주차 복습**: 범주형 변수 간 관계 분석 (카이자승 검정)
- **이번 주 질문**: **연속형 변수**(소득, 점수, 만족도 등)의 집단 간 차이 비교
  - 사회과학에서 매우 흔한 질문
- **이번 주 학습 도구**: **t-검정**과 **ANOVA**

---

## Part 1. 독립표본 t-검정

### 핵심 개념 1: 독립표본 t-검정(independent samples t-test)

- **독립표본 t-검정(independent samples t-test)** 정의
  - 두 독립 집단의 평균이 통계적으로 유의하게 다른지를 검정하는 방법

- **핵심 직관**:

> t = (두 집단의 평균 차이) ÷ (그 차이의 표준오차)

- **t값이 커지는 조건**:
  - **분자**(평균 차이)가 크거나
  - **분모**(변동)가 작으면
  - → t값 증가 → 유의할 가능성 높아짐

**실행 전 확인: 등분산 가정**

- t-검정은 두 집단의 **분산이 같다**고 가정
- **Levene 검정**으로 확인
  - p ≥ 0.05 → 등분산 가정 충족 → `stats.ttest_ind(equal_var=True)`
  - p < 0.05 → 등분산 가정 위반 → **Welch's t-검정** 사용 (`equal_var=False`)

---

#### 예측 투표 — 등분산 위반 시 어떻게?

> **투표 질문**: Levene 검정 결과 p = 0.02가 나왔다. 어떻게 해야 하는가?
>
> (가) 그냥 t-검정을 실행한다
> (나) Welch's t-검정(`equal_var=False`)을 사용한다
> (다) t-검정을 포기하고 카이자승 검정을 한다
>
> *(10초)*

- **정답: (나)**
  - p < 0.05 → 등분산 가정 위반
  - **Welch's t-검정**은 등분산을 가정하지 않으므로 안전한 대안
  - `stats.ttest_ind(male, female, equal_var=False)`

---

### t-검정 효과 크기: Cohen's d

- p값만으로는 차이의 **크기**를 알 수 없음 → 효과 크기 함께 보고 필요

| 지표 | 작은 효과 | 중간 효과 | 큰 효과 |
|---|---:|---:|---:|
| **Cohen's d** | 약 0.2 | 약 0.5 | 약 0.8+ |

---

### 실습 — 성별 × 소득 (t-검정)

- social_survey_w06.csv 데이터에서 성별에 따른 소득 차이를 t-검정으로 분석한다.

In [16]:
from scipy import stats
import pandas as pd
import numpy as np

df = pd.read_csv("data/social_survey_w06.csv", encoding='utf-8')
male = df[df["성별"] == "남성"]["소득"]
female = df[df["성별"] == "여성"]["소득"]

# 등분산 검정
lev_stat, lev_p = stats.levene(male, female)
print(f"=== 등분산 검정 ===")
print(f"Levene 검정: p = {lev_p:.4f}")
if lev_p >= 0.05:
    print("→ 등분산 가정 충족 (equal_var=True)")
else:
    print("→ 등분산 가정 위반 → Welch's t-검정 사용 (equal_var=False)")

# t-검정 실행
equal_var = lev_p >= 0.05
t_stat, p_val = stats.ttest_ind(male, female, equal_var=equal_var)
d = (male.mean() - female.mean()) / np.sqrt(((len(male)-1)*male.std()**2 + (len(female)-1)*female.std()**2) / (len(male)+len(female)-2))

print(f"\n=== t-검정 결과 ===")
print(f"t({len(male)+len(female)-2}) = {t_stat:.2f}, p = {p_val:.4f}")
print(f"Cohen's d = {d:.2f}")
print(f"\n=== 기술통계 ===")
print(f"남성 평균: {male.mean():.0f}만 원 (SD = {male.std():.0f})")
print(f"여성 평균: {female.mean():.0f}만 원 (SD = {female.std():.0f})")

=== 등분산 검정 ===
Levene 검정: p = 0.6973
→ 등분산 가정 충족 (equal_var=True)

=== t-검정 결과 ===
t(198) = 2.39, p = 0.0178
Cohen's d = 0.34

=== 기술통계 ===
남성 평균: 359만 원 (SD = 153)
여성 평균: 310만 원 (SD = 140)


**결과 해석:**

- **등분산 검정**:
  - Levene 검정: p > 0.05 → 등분산 충족
- **t-검정 결과**:
  - **t(198) = 2.39, p = 0.0178**
  - **Cohen's d = 0.34** (작은~중간 수준 효과)
- **기술통계**:
  - 남성 평균: 359만 원
  - 여성 평균: 310만 원

- 유의수준 0.05에서, 남성과 여성의 평균 소득에는 **통계적으로 유의한 차이** 있음
- **⚠️ 주의**: 직종/경력/근무시간 등 교란변수 미통제 → 성별이 소득을 "결정한다"는 뜻 아님

- _전체 코드는 practice/W06/code/W06-1-t-test-anova.py 참고_

---

### 과제 — t-검정 (15분)

> Copilot에게 다음과 같이 요청한다.
>
> "data/social_survey_w06.csv에서 성별(남성/여성) 두 집단의 소득 평균 차이를 검정하는 코드를 작성해줘. 등분산 검정(Levene), 독립표본 t-검정, Cohen's d 계산, 그리고 성별에 따른 소득 박스플롯까지 포함하세요"

In [17]:
# 아래에 코드를 붙여넣는다.


#### 과제 해석

> t-검정 결과를 해석문 4단계 구조로 작성하세요.
>
> **(분석목적)** ___를 보기 위해 ___를 사용했다.
>
> **(핵심근거)** 출력에서 ___(t값, p값, Cohen's d, 등분산 충족 여부, 각 집단의 평균과 SD)를 확인했다.
>
> **(결론)** 따라서 ___이라고 말할 수 있다.
>
> **(주의)** 이 결과는 ___를 의미하지 않으며, ___의 한계가 있다.

#### 짝 점검 — t-검정

> 옆 사람과 서로의 결과를 비교한다.
>
> 1. 서로의 **t값, p값, Cohen's d**가 동일한지 확인한다.
> 2. **등분산 검정**(Levene)을 수행했는지 점검한다.
> 3. 해석문에서 **4단계가 모두 포함**되어 있는지, **인과 표현**이 없는지 확인한다.
>
> 결과가 다르면 코드를 함께 점검하고, 해석문에 빠진 부분이 있으면 1문장으로 보충 제안한다.

---

## Part 2. ANOVA (분산분석)

### 핵심 개념 2: ANOVA(Analysis of Variance)

- **ANOVA(Analysis of Variance, 분산분석)** 정의
  - 세 집단 이상의 평균이 모두 같은지를 한꺼번에 검정하는 방법
  - 귀무가설: "모든 집단의 모평균이 동일하다"
  - 기각 시: "적어도 하나의 집단 평균이 다르다"고 판단

- **중요 질문: "왜 t-검정을 여러 번 하면 안 되는가?"**
  - 3개 집단 비교 시 t-검정 3번 실행 → 전체 1종 오류율 약 14.3%까지 상승
  - 이를 **다중비교 문제(multiple comparison problem)**라 함
  - ANOVA는 한 번의 F-검정으로 이 문제 해결

- **F값의 의미**:

> F = 집단 간 변동(between) ÷ 집단 내 변동(within)

- 집단 간 차이가 집단 내 변동에 비해 충분히 크면 → F값 증가 → 귀무가설 기각

---

### 핵심 개념 3: 사후검정(post-hoc test)

- **ANOVA F-검정이 유의하면**
  - "적어도 하나의 집단 쌍이 다르다"는 것만 알 수 있음
  - **어느 집단 쌍이 다른지**는 **사후검정(post-hoc test)**으로 확인

- **주요 사후검정 방법**:
  - **Tukey HSD(Honestly Significant Difference)**: 모든 집단 쌍을 비교하면서 다중비교 보정 적용
  - **Bonferroni 보정**: 유의수준(α)을 비교 횟수로 나누어 더 엄격한 기준 적용

> **주의**: ANOVA가 유의하다고 해서 "**모든** 집단이 서로 다르다"는 뜻이 아님

---

#### 오답 수집 — "ANOVA 유의 = 모든 집단이 다르다?"

> **질문**: ANOVA 결과 F(2, 197) = 18.75, p < 0.001이 나왔다. 다음 중 올바른 해석은?
>
> ① 세 집단의 평균이 모두 서로 다르다
> ② 적어도 하나의 집단 쌍에서 평균 차이가 유의하다
> ③ 첫 번째 집단의 평균이 가장 높다
>
> *(오답을 고른 사람이 있다면, 왜 ①이 틀린지 옆 사람에게 설명해 보자. 30초)*

- **정답: ②**
  - ANOVA의 F-검정 = "적어도 하나의 쌍이 다르다"만 알려줌
  - 어느 쌍인지, 모든 쌍이 다 다른지 → **사후검정으로 확인** 필요

---

### ANOVA 효과 크기: η²

| 지표 | 작은 효과 | 중간 효과 | 큰 효과 |
|---|---:|---:|---:|
| **η²** | 약 0.01 | 약 0.06 | 약 0.14+ |

---

### 실습 — 교육수준 × 소득 (ANOVA)

- social_survey_w06.csv 데이터에서 교육수준(고졸/대졸/대학원)에 따른 소득 차이를 ANOVA로 분석한다.

In [18]:
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import pandas as pd
import numpy as np

df = pd.read_csv("data/social_survey_w06.csv", encoding='utf-8')
high = df[df["교육수준"] == "고졸"]["소득"]
univ = df[df["교육수준"] == "대졸"]["소득"]
grad = df[df["교육수준"] == "대학원"]["소득"]

# 등분산 검정
lev_stat, lev_p = stats.levene(high, univ, grad)
print("=== 등분산 검정 ===")
print(f"Levene 검정: p = {lev_p:.4f}")
if lev_p >= 0.05:
    print("→ 등분산 가정 충족")
else:
    print("→ 등분산 가정 위반 → Welch ANOVA 고려")

# ANOVA 실행
f_stat, f_p = stats.f_oneway(high, univ, grad)

# η² 계산
groups = [high, univ, grad]
grand_mean = df["소득"].mean()
ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups)
ss_total = sum((df["소득"] - grand_mean)**2)
eta_sq = ss_between / ss_total

print(f"\n=== ANOVA 결과 ===")
print(f"F(2, {len(df)-3}) = {f_stat:.2f}, p = {f_p:.4f}")
print(f"η² = {eta_sq:.3f}")

print(f"\n=== 기술통계 ===")
print(f"고졸 평균: {high.mean():.0f}만 원 (SD = {high.std():.0f})")
print(f"대졸 평균: {univ.mean():.0f}만 원 (SD = {univ.std():.0f})")
print(f"대학원 평균: {grad.mean():.0f}만 원 (SD = {grad.std():.0f})")

# Tukey HSD 사후검정
tukey = pairwise_tukeyhsd(df["소득"], df["교육수준"], alpha=0.05)
print(f"\n=== Tukey HSD 사후검정 ===")
print(tukey)

=== 등분산 검정 ===
Levene 검정: p = 0.6074
→ 등분산 가정 충족

=== ANOVA 결과 ===
F(2, 197) = 19.58, p = 0.0000
η² = 0.166

=== 기술통계 ===
고졸 평균: 269만 원 (SD = 133)
대졸 평균: 330만 원 (SD = 128)
대학원 평균: 421만 원 (SD = 149)

=== Tukey HSD 사후검정 ===
 Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj   lower   upper   reject
-----------------------------------------------------
    고졸     대졸  61.4955 0.0203  7.7771 115.2139   True
    고졸    대학원 151.5835    0.0 94.2932 208.8737   True
    대졸    대학원   90.088 0.0007 33.3285 146.8474   True
-----------------------------------------------------


**결과 해석:**

- **등분산 검정**:
  - Levene 검정: p > 0.05 → 등분산 충족
- **ANOVA 결과**:
  - F값이 유의하면 → "적어도 하나의 집단 쌍에서 평균 차이가 유의하다"
  - η² 값으로 효과 크기 판단 (0.01 = 작은, 0.06 = 중간, 0.14+ = 큰)
- **사후검정**:
  - Tukey HSD로 **어느 집단 쌍**이 유의하게 다른지 확인
  - reject = True인 쌍만 유의한 차이가 있다고 판단

- **⚠️ 주의**: ANOVA 유의 ≠ 모든 집단이 다르다. 사후검정 결과를 반드시 확인해야 함

---

### 과제 — ANOVA (15분)

> Copilot에게 다음과 같이 요청한다.
>
> "data/social_survey_w06.csv에서 교육수준(고졸/대졸/대학원) 세 집단의 소득 평균 차이를 검정하는 코드를 작성해줘. 등분산 검정(Levene), 일원배치 ANOVA(F검정), η² 계산, Tukey HSD 사후검정, 그리고 교육수준별 소득 박스플롯까지 포함하세요"

In [19]:
# 아래에 코드를 붙여넣는다.


#### 과제 해석

> ANOVA 결과를 해석문 4단계 구조로 작성하세요.
>
> **(분석목적)** ___를 보기 위해 ___를 사용했다.
>
> **(핵심근거)** 출력에서 ___(F값, p값, η², 각 집단의 평균과 SD, Tukey HSD 사후검정 결과)를 확인했다.
>
> **(결론)** 따라서 ___이라고 말할 수 있다.
>
> **(주의)** 이 결과는 ___를 의미하지 않으며, ___의 한계가 있다.

#### 짝 점검 — ANOVA

> 옆 사람과 서로의 결과를 비교한다.
>
> 1. 서로의 **F값, p값, η²**가 동일한지 확인한다.
> 2. **Tukey HSD 사후검정** 결과가 일치하는지 점검한다.
> 3. 해석문에서 **4단계가 모두 포함**되어 있는지, **"모든 집단이 다르다"는 과잉 해석**이 없는지 확인한다.
>
> 결과가 다르면 코드를 함께 점검하고, 해석문에 빠진 부분이 있으면 1문장으로 보충 제안한다.

---

### 비판적 읽기 (5–10분)

- **가상의 뉴스 기사 발췌문**

> **〈○○일보〉 2026.03.10**
>
> *"국내 직장인 800명을 대상으로 조사한 결과, 대학원 졸업자의 평균 연봉이 고졸 이하보다 1,200만 원 높은 것으로 나타나, **교육이 소득을 높이는 핵심 요인으로 확인되었다**. 연구진은 학력이 높을수록 소득이 올라간다고 결론지었다."*

> **토론 질문**: 이 기사의 해석에서 문제점을 찾아보자. (2–3분, 짝 토론 후 전체 공유)

**문제점 분석:**

1. **인과 표현**
   - 문제: "교육이 소득을 **높이는** 핵심 요인"이라고 서술
   - 사실: 횡단면 조사로는 **연관만** 확인 가능
   - 인과 주장 요건: 실험이나 종단 연구 필요
   - 적절한 표현: "교육수준에 따라 소득에 차이가 **관찰되었다**"

2. **효과 크기 미보고**
   - 문제: p값이나 효과 크기(Cohen's d, η²) 전혀 보고 안 됨
   - 주의점: 800명 큰 표본에서는 사소한 차이도 유의하게 나올 수 있음
   - 결과: 효과 크기 없이는 실질적 의미 판단 불가

3. **교란변수 무시**
   - 문제: 연령, 직종, 근무시간, 부모 소득 등 미통제
   - 이 변수들: 교육수준과 소득 **모두에** 관련될 수 있음
   - 결과: "학력이 높을수록 소득이 올라간다"는 단순화된 결론 = **과잉 해석**